In [1]:
from datasets import load_dataset

ds = load_dataset("mteb/summeval", split="test")
print(ds.features)
print(ds[0].keys())

{'machine_summaries': List(Value('string')), 'human_summaries': List(Value('string')), 'relevance': List(Value('float64')), 'coherence': List(Value('float64')), 'fluency': List(Value('float64')), 'consistency': List(Value('float64')), 'text': Value('string'), 'id': Value('string')}
dict_keys(['machine_summaries', 'human_summaries', 'relevance', 'coherence', 'fluency', 'consistency', 'text', 'id'])


In [2]:
import pandas as pd
from transformers import AutoTokenizer

# Samakan dengan model dan konfigurasi eksperimen.
TOKENIZER_NAME = "LiquidAI/LFM2.5-2.6B"
CONTEXT_LENGTH = 32768
MAX_OUTPUT_TOKENS = 2048
SAFETY_MARGIN = 256  # Pilihan margin, bukan batas bawaan model.

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

instruction = (
    "Summarize the article in 2 to 3 concise sentences. "
    "Use only facts from the article."
)

records = []
for row in ds:
    messages = [
        {
            "role": "user",
            "content": instruction + "\n\nArticle:\n" + row["text"],
        }
    ]
    # Kalau request aslinya memakai system message, tambahkan juga di sini.
    prompt_tokens = len(tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
    ))

    records.append({
        "id": row["id"],
        "source_tokens": len(tokenizer.encode(
            row["text"], add_special_tokens=False
        )),
        "prompt_tokens": prompt_tokens,
        "needs_chunking": (
            prompt_tokens + MAX_OUTPUT_TOKENS + SAFETY_MARGIN
            > CONTEXT_LENGTH
        ),
    })

token_profile = pd.DataFrame(records)

display(token_profile[["source_tokens", "prompt_tokens"]].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
).round(1))

print(
    f"Artikel yang melampaui budget: "
    f"{token_profile['needs_chunking'].sum()}/{len(token_profile)}"
)
display(token_profile.nlargest(5, "prompt_tokens"))

,source_tokens,prompt_tokens
count,100.0,100.0
mean,462.8,2.0
std,130.5,0.0
min,132.0,2.0
50%,477.5,2.0
90%,613.3,2.0
95%,652.2,2.0
99%,666.6,2.0
max,730.0,2.0


Artikel yang melampaui budget: 0/100


,id,source_tokens,prompt_tokens,needs_chunking
0,cnn-test-404f859482d47c127868964a9a39d1a7645dd2e9,537,2,False
1,cnn-test-4761dc6d8bdf56b9ada97104113dd1bcf4aed3f1,381,2,False
2,cnn-test-5139ccfabee55ddb83e7937f5802c0a67aee8975,373,2,False
3,cnn-test-88c2481234e763c9bbc68d0ab1be1d2375c1349a,339,2,False
4,cnn-test-a02e362c5b8f049848ce718b37b96117485461cf,343,2,False


In [2]:
import pandas as pd

score_columns = ["relevance", "coherence", "fluency", "consistency"]

profile = pd.DataFrame({
    "id": ds["id"],
    "source_words": [len(text.split()) for text in ds["text"]],
    "machine_summary_count": [len(x) for x in ds["machine_summaries"]],
    "human_summary_count": [len(x) for x in ds["human_summaries"]],
})

display(profile.describe().round(1))
display(profile.head())

,source_words,machine_summary_count,human_summary_count
count,100.0,100.0,100.0
mean,359.3,16.0,11.0
std,103.1,0.0,0.0
min,100.0,16.0,11.0
25%,281.8,16.0,11.0
50%,367.0,16.0,11.0
75%,444.5,16.0,11.0
max,540.0,16.0,11.0


,id,source_words,machine_summary_count,human_summary_count
0,cnn-test-404f859482d47c127868964a9a39d1a7645dd2e9,385,16,11
1,cnn-test-4761dc6d8bdf56b9ada97104113dd1bcf4aed3f1,293,16,11
2,cnn-test-5139ccfabee55ddb83e7937f5802c0a67aee8975,281,16,11
3,cnn-test-88c2481234e763c9bbc68d0ab1be1d2375c1349a,263,16,11
4,cnn-test-a02e362c5b8f049848ce718b37b96117485461cf,282,16,11


In [3]:
i = 0
row = ds[i]

print(f"ID: {row['id']}")
print(f"Source ({len(row['text'].split())} words):\n{row['text'][:1500]}...\n")

print(f"Human references: {len(row['human_summaries'])}")
for n, summary in enumerate(row["human_summaries"], start=1):
    print(f"\nReference {n}:\n{summary}")

print(f"\nMachine summaries: {len(row['machine_summaries'])}")
for n, summary in enumerate(row["machine_summaries"], start=1):
    scores = ", ".join(
        f"{metric}={row[metric][n - 1]:.2f}"
        for metric in score_columns
    )
    print(f"\nCandidate {n} ({scores}):\n{summary}")

ID: cnn-test-404f859482d47c127868964a9a39d1a7645dd2e9
Source (385 words):
(CNN)Donald Sterling's racist remarks cost him an NBA team last year. But now it's his former female companion who has lost big. A Los Angeles judge has ordered V. Stiviano to pay back more than $2.6 million in gifts after Sterling's wife sued her. In the lawsuit, Rochelle "Shelly" Sterling accused Stiviano of targeting extremely wealthy older men. She claimed Donald Sterling used the couple's money to buy Stiviano a Ferrari, two Bentleys and a Range Rover, and that he helped her get a $1.8 million duplex. Who is V. Stiviano? Stiviano countered that there was nothing wrong with Donald Sterling giving her gifts and that she never took advantage of the former Los Angeles Clippers owner, who made much of his fortune in real estate. Shelly Sterling was thrilled with the court decision Tuesday, her lawyer told CNN affiliate KABC. "This is a victory for the Sterling family in recovering the $2,630,000 that Donald lavis

In [4]:
records = []
mismatches = []

for row in ds:
    n_candidates = len(row["machine_summaries"])
    lengths = {metric: len(row[metric]) for metric in score_columns}

    if any(n != n_candidates for n in lengths.values()):
        mismatches.append({
            "id": row["id"],
            "machine_summaries": n_candidates,
            **lengths,
        })
        continue

    for j, summary in enumerate(row["machine_summaries"]):
        records.append({
            "id": row["id"],
            "candidate_index": j,
            "summary": summary,
            **{metric: row[metric][j] for metric in score_columns},
        })

candidate_df = pd.DataFrame(records)

print(f"Candidate summaries: {len(candidate_df)}")
print(f"Rows with score-count mismatches: {len(mismatches)}")

display(candidate_df[score_columns].describe().round(2))
display(candidate_df[score_columns].corr().round(2))

Candidate summaries: 1600
Rows with score-count mismatches: 0


,relevance,coherence,fluency,consistency
count,1600.00,1600.00,1600.00,1600.00
mean,3.78,3.41,4.67,4.66
std,0.80,1.04,0.73,0.92
min,1.00,1.00,1.00,1.00
25%,3.33,2.67,4.67,5.00
50%,4.00,3.67,5.00,5.00
75%,4.33,4.33,5.00,5.00
max,5.00,5.00,5.00,5.00


,relevance,coherence,fluency,consistency
relevance,1.00,0.66,0.37,0.41
coherence,0.66,1.00,0.38,0.32
fluency,0.37,0.38,1.00,0.49
consistency,0.41,0.32,0.49,1.00


In [5]:
i = 0
row = ds[i]

print(f"ID: {row['id']}")
print(f"Source ({len(row['text'].split())} words):\n{row['text'][:1500]}...\n")

print(f"Human references: {len(row['human_summaries'])}")
for n, summary in enumerate(row["human_summaries"], start=1):
    print(f"\nReference {n}:\n{summary}")

print(f"\nMachine summaries: {len(row['machine_summaries'])}")
for n, summary in enumerate(row["machine_summaries"], start=1):
    scores = ", ".join(
        f"{metric}={row[metric][n - 1]:.2f}"
        for metric in score_columns
    )
    print(f"\nCandidate {n} ({scores}):\n{summary}")

ID: cnn-test-404f859482d47c127868964a9a39d1a7645dd2e9
Source (385 words):
(CNN)Donald Sterling's racist remarks cost him an NBA team last year. But now it's his former female companion who has lost big. A Los Angeles judge has ordered V. Stiviano to pay back more than $2.6 million in gifts after Sterling's wife sued her. In the lawsuit, Rochelle "Shelly" Sterling accused Stiviano of targeting extremely wealthy older men. She claimed Donald Sterling used the couple's money to buy Stiviano a Ferrari, two Bentleys and a Range Rover, and that he helped her get a $1.8 million duplex. Who is V. Stiviano? Stiviano countered that there was nothing wrong with Donald Sterling giving her gifts and that she never took advantage of the former Los Angeles Clippers owner, who made much of his fortune in real estate. Shelly Sterling was thrilled with the court decision Tuesday, her lawyer told CNN affiliate KABC. "This is a victory for the Sterling family in recovering the $2,630,000 that Donald lavis

In [6]:
records = []
mismatches = []

for row in ds:
    n_candidates = len(row["machine_summaries"])
    lengths = {metric: len(row[metric]) for metric in score_columns}

    if any(n != n_candidates for n in lengths.values()):
        mismatches.append({
            "id": row["id"],
            "machine_summaries": n_candidates,
            **lengths,
        })
        continue

    for j, summary in enumerate(row["machine_summaries"]):
        records.append({
            "id": row["id"],
            "candidate_index": j,
            "summary": summary,
            **{metric: row[metric][j] for metric in score_columns},
        })

candidate_df = pd.DataFrame(records)

print(f"Candidate summaries: {len(candidate_df)}")
print(f"Rows with score-count mismatches: {len(mismatches)}")

display(candidate_df[score_columns].describe().round(2))
display(candidate_df[score_columns].corr().round(2))

Candidate summaries: 1600
Rows with score-count mismatches: 0


,relevance,coherence,fluency,consistency
count,1600.00,1600.00,1600.00,1600.00
mean,3.78,3.41,4.67,4.66
std,0.80,1.04,0.73,0.92
min,1.00,1.00,1.00,1.00
25%,3.33,2.67,4.67,5.00
50%,4.00,3.67,5.00,5.00
75%,4.33,4.33,5.00,5.00
max,5.00,5.00,5.00,5.00


,relevance,coherence,fluency,consistency
relevance,1.00,0.66,0.37,0.41
coherence,0.66,1.00,0.38,0.32
fluency,0.37,0.38,1.00,0.49
consistency,0.41,0.32,0.49,1.00


In [7]:
low_scoring = candidate_df.nsmallest(5, "relevance").copy()
low_scoring["preview"] = low_scoring["summary"].str.slice(0, 300)

display(low_scoring[["id", "relevance", "coherence", "fluency", "consistency", "preview"]])

,id,relevance,coherence,fluency,consistency,preview
160,cnn-test-e2bccd4dec93c9bb7b327827dae004c2d494ec31,1.0,1.000000,1.333333,1.000000,video game `` space invaders '' was developed ...
173,cnn-test-e2bccd4dec93c9bb7b327827dae004c2d494ec31,1.0,1.333333,1.333333,1.000000,`` space invaders '' is the founder of the new...
175,cnn-test-e2bccd4dec93c9bb7b327827dae004c2d494ec31,1.0,1.000000,2.000000,2.666667,`` space invaders '' was developed in japan ba...
1504,dm-test-eeef09d26cf30c2124c0399b08eedc6321fe5d20,1.0,1.666667,5.000000,1.000000,real madrid beat chile 1-0 in el clasico on th...
1518,dm-test-eeef09d26cf30c2124c0399b08eedc6321fe5d20,1.0,1.666667,5.000000,5.000000,neymar helped brazil beat chile 1-0 at the emi...


## Generate and score one summary

Use the first Summeval article, generate one summary with the configured OpenRouter model, and score it against all 11 human references. BERTScore's multiple-reference mode reports the best matching reference for each candidate, so the result is a best-match F1.


In [8]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

PROJECT_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project root or one of its subfolders.")

load_dotenv(PROJECT_ROOT / ".env")
model_name = os.getenv("OPENROUTER_MODEL", "").strip()
if not model_name:
    raise ValueError("Set OPENROUTER_MODEL in .env, for example provider/model-name.")

llm = ChatOpenRouter(model=model_name, temperature=0)
row = ds[0]
response = llm.invoke(
    "Summarize the article in 2 to 3 concise sentences. Use only facts from the article."
    + "\n\nArticle:\n"
    + row["text"]
)
machine_summary = response.content.strip()

print(f"ID: {row['id']}")
print(f"Model: {model_name}")
print(f"Human references: {len(row['human_summaries'])}")
print(f"Example human reference:\n{row['human_summaries'][0]}")
print(f"\nGenerated summary:\n{machine_summary}")


d:\Workspace\Repository\llm-summarization\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


ID: cnn-test-404f859482d47c127868964a9a39d1a7645dd2e9
Model: nvidia/nemotron-3-ultra-550b-a55b:free
Human references: 11
Example human reference:
V. Stiviano must pay back $2.6 million in gifts from Donald Sterling. Sterling's wife claimed the ex-Clippers used the couple's money for the gifts. The items included a Ferrari, two Bentleys and a Range Rover.

Generated summary:
A Los Angeles judge ordered V. Stiviano to repay over $2.6 million in gifts—including luxury cars and a duplex—after Shelly Sterling sued, alleging her husband used community property to lavish the items on Stiviano. The court ruled in Shelly Sterling's favor, establishing a precedent allowing a spouse to recover damages from a gift recipient. Donald Sterling was previously banned from the NBA and forced to sell the Clippers following the release of racist remarks he made to Stiviano on an audio recording.


Install the direct BERTScore package in the active notebook environment, then run the scoring cell. The first run may download the English encoder model.


In [9]:
from bert_score import score

precision, recall, f1 = score(
    cands=[machine_summary],
    refs=[row["human_summaries"]],
    lang="en",
    verbose=True,
)

bertscore_result = {
    # "precision": precision.item(),
    # "recall": recall.item(),
    "f1": f1.item(),
}
bertscore_result


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.73 seconds, 6.35 sentences/sec


{'f1': 0.9007304906845093}

In [10]:
from transformers import AutoModelForSequenceClassification

hhem = AutoModelForSequenceClassification.from_pretrained(
    "vectara/hallucination_evaluation_model",
    trust_remote_code=True,
)

hhem_score = hhem.predict([
    (row["text"], machine_summary),
]).item()

evaluation_result = {
    "bertscore_f1": bertscore_result["f1"],
    "hhem_consistency": hhem_score,
}

evaluation_result

configuration_hhem_v2.py:   0%|          | 0.00/760 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/vectara/hallucination_evaluation_model:
- configuration_hhem_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] You are using a model of type `HHEMv2Config` to instantiate a model of type `HHEMv2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


modeling_hhem_v2.py:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/vectara/hallucination_evaluation_model:
- modeling_hhem_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

...

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/113 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (712 > 512). Running this sequence through the model will result in indexing errors


{'bertscore_f1': 0.9007304906845093, 'hhem_consistency': 0.9414041638374329}